In [47]:
# goal: prototype skeleton transformer
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from map4 import MAP4
import re

import matplotlib.pyplot as plt
%matplotlib inline

from pathlib import Path

In [48]:
from src.nano_maker.modules.nAAno.radialseeker import RadialSeeker
from src.nano_maker.modules.nano_printer.smiles_handler import smiles_fingerprint
from src.database.dataset import RadialDataset

In [49]:
# Load data
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
data_folder = Path("notebook_database")
RADIAL_SEQUENCES = pd.read_pickle(data_folder/"radial_seq_df.pkl")
MOLECULAR_FINGERPRINTS = pd.read_pickle(data_folder/"molfp_df.pkl")
TRAIN_POINTER = pd.read_parquet(data_folder/"training_pointers.parquet")
TEST_POINTER = pd.read_parquet(data_folder/"test_pointers.parquet")

In [50]:
print(len(TRAIN_POINTER))
print(len(TEST_POINTER))

14120350
3630380


In [51]:
print(MOLECULAR_FINGERPRINTS.head(3))
print(RADIAL_SEQUENCES.head(3))

sample_fp = MOLECULAR_FINGERPRINTS["map4_fp"].iloc[0]
print(len(sample_fp), sample_fp[0].dtype)

                                          smiles_str  \
0  C[C@@H](N1CC[C@](C)(C1=O)c1ccc(OCc2cc(C)nc3ccc...   
1  Cc1ccc(CNc2cc(OC[C@H]3C[C@@H]3c3ccc4CCCc4n3)nn...   
2  C[C@@H](COc1ccnc2CCC[C@@H](C)c12)C[C@H]1Cc2cc3...   

                                             map4_fp  
0  [tensor(0.), tensor(1.), tensor(1.), tensor(1....  
1  [tensor(0.), tensor(1.), tensor(0.), tensor(0....  
2  [tensor(0.), tensor(1.), tensor(1.), tensor(1....  
  PDB_ID                                    radial_sequence
0   5UEY  [[[I], [61, 56, 32]], [[K], [68, 53, 41]], [[M...
1   6FID  [[[Y], [46, 67, 40]], [[D], [67, 59, 52]], [[H...
2   1WCC  [[[L], [35, 47, 36]], [[K], [65, 61, 44]], [[V...
1024 torch.float32


In [52]:
sample_train_pointer = TRAIN_POINTER.sample(10000)
sample_test_pointer = TEST_POINTER.sample(1000)

In [53]:
# RadialSeeker parameters used -> record these - might design a config file later
radial_resolution = 100
intrashell_resolution = 100
max_angstroms = 33

block_size = 50
batch_size = 100
dropout=0.1
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# training dataset
training_dataset = RadialDataset(pointer=sample_train_pointer,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size)
# DataLoader
from torch.utils.data import DataLoader
loader = DataLoader(
    training_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last=True
)

torch.manual_seed(311104)

# test / validation set
test_set = RadialDataset(pointer=sample_test_pointer,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size)
# DataLoader
from torch.utils.data import DataLoader
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=4,
    shuffle=False,
    drop_last=True
)

In [70]:
class SkeletonModel(nn.Module):
    def __init__(self, n_embd, n_head, block_size, map4_res, radial_resolution):
        super().__init__()
        self.block_size = block_size
        self.map4_res = map4_res
        self.radial_resolution = radial_resolution

        self.c_project = nn.Linear(3, n_embd)  # feed coordinates here
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.mol_encoder = nn.Sequential(
            nn.Linear(map4_res, int(map4_res//2)),
            nn.ReLU(),
            nn.Linear(int(map4_res//2), n_embd),
            nn.LayerNorm(n_embd)
        )


        self.stack = Stack(n_embd, n_head)

        # OUTPUT HEAD -> outputs coordinates
        self.c_head = nn.Linear(n_embd, 3)


    def forward(self, coord_hist, map4_enc, targets=None):
        """
        coord_hist will be [n_batch, block_size, 3]
        targets is [n_batch, 1, 3]
        map4_enc -> unsure how I'm going to encode this
        """
        B, T, C = coord_hist.shape
        coordinate_emb = self.c_project(coord_hist.float() / self.radial_resolution)
        pos_emb = self.pos_emb(torch.arange(T, device=coord_hist.device))
        x = coordinate_emb + pos_emb

        mol_emb = self.mol_encoder(map4_enc.float()).unsqueeze(1)

        x = self.stack(x, mol_emb)

        output_coords = self.c_head(x[:, -1, :])

        loss = None
        if targets is not None:
            loss = F.mse_loss(output_coords, (targets.float() / self.radial_resolution))

        return output_coords, loss

    def generate(self, map4_enc, max_steps=130):
        # largest protein pocket in dataset was 107
        map4_enc = map4_enc.to(device)
        coord_context = torch.zeros(1, block_size, 3, device=map4_enc.device)
        coord_out = []

        for _ in range(max_steps):
            next_coord, _ = self.forward(coord_context, map4_enc)
            coord_out.append(next_coord)
            coord_context = torch.cat([coord_context[:, 1:, :], next_coord.unsqueeze(1)], dim=1)


        return coord_out

In [55]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        # compute affinities
        attn_wts = q @ k.transpose(-2, -1) * C**-0.5
        attn_wts = F.softmax(attn_wts, dim=-1)
        attn_wts = self.dropout(attn_wts)

        # weighed aggregation
        values = self.value(x)
        output = attn_wts @ values
        return output


class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out

In [56]:
class CrossAttentionHead(nn.Module):
    def __init__(self, n_embd, head_size, dropout=0.2):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coordinate_emb, molecular_emb):
        q = self.query(coordinate_emb)
        k = self.key(molecular_emb)
        v = self.value(molecular_emb)

        head_size = q.shape[-1]

        c_attn = q @ k.transpose(-2, -1) * head_size**-0.5
        c_attn = F.softmax(c_attn, dim=-1)
        c_attn = self.dropout(c_attn)

        return c_attn @ v


class MultiHeadCrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.2):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([CrossAttentionHead(n_embd,
                                                       head_size,
                                                       ) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coord_emb, mol_emb):
        out = torch.cat([h(coord_emb, mol_emb) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out

In [57]:
class FeedForwardNet(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [58]:
class Stack(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.satt_head = MultiHeadAttention(n_embd, n_head)
        self.ffwd = FeedForwardNet(n_embd)
        self.catt_head = MultiHeadCrossAttention(n_embd, n_head)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, mol_emb):
        x = x + self.satt_head(self.ln1(x))
        x = x + self.catt_head(self.ln2(x), mol_emb)
        x = x + self.ffwd(self.ln3(x))
        return x

In [59]:
# TODO: build small MLP for MAP4 encoding

In [60]:
model = SkeletonModel(n_embd=256, n_head=4, block_size=50, map4_res=1024, radial_resolution=100).to(device)

In [61]:
learning_rate = 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
model.train()

SkeletonModel(
  (c_project): Linear(in_features=3, out_features=256, bias=True)
  (pos_emb): Embedding(50, 256)
  (mol_encoder): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (stack): Stack(
    (satt_head): MultiHeadAttention(
      (heads): ModuleList(
        (0-3): 4 x Head(
          (key): Linear(in_features=256, out_features=64, bias=False)
          (query): Linear(in_features=256, out_features=64, bias=False)
          (value): Linear(in_features=256, out_features=64, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (projection): Linear(in_features=256, out_features=256, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (ffwd): FeedForwardNet(
      (net): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): R

In [62]:
@torch.no_grad()
def estimate_loss(model, loader, device, radial_res, max_batches=None):
    model.eval()
    total_loss = 0
    n_batches = 0

    for batch_idx, batch in enumerate(loader):
        if max_batches and batch_idx >= max_batches:
            break

        map4_fp, radial_X, radial_Y = batch

        map4_fp = map4_fp.to(device)
        radial_X = radial_X.to(device)
        radial_Y = radial_Y.to(device)

        _, loss = model(radial_X, map4_fp, radial_Y)
        total_loss += loss.item()
        n_batches += 1

    model.train()
    return total_loss / n_batches


In [63]:
for epoch in range(10):   # start with few epochs
    total_loss = 0
    for batch_idx, batch in enumerate(loader):
        map4_fp, radial_X, radial_Y = batch

        map4_fp = map4_fp.to(device)
        radial_X = radial_X.to(device)
        radial_Y = radial_Y.to(device)

        optimizer.zero_grad()
        pred, loss = model(radial_X, map4_fp, radial_Y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch + 1} | Batch {batch_idx + 1} | Loss: {loss.item():.6f}")

    train_loss = total_loss / len(loader)
    val_loss = estimate_loss(model, test_loader, device, radial_res=radial_resolution, max_batches=batch_size)

    print(f"Epoch {epoch+1} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Gap: {val_loss - train_loss:.6f}")


Epoch 1 | Batch 1 | Loss: 1.597293
Epoch 1 | Batch 51 | Loss: 0.032014
Epoch 1 | Train: 0.158545 | Val: 0.015197 | Gap: -0.143348
Epoch 2 | Batch 1 | Loss: 0.010428
Epoch 2 | Batch 51 | Loss: 0.014000
Epoch 2 | Train: 0.015884 | Val: 0.015198 | Gap: -0.000686
Epoch 3 | Batch 1 | Loss: 0.009313
Epoch 3 | Batch 51 | Loss: 0.017486
Epoch 3 | Train: 0.015420 | Val: 0.015937 | Gap: 0.000518
Epoch 4 | Batch 1 | Loss: 0.014664
Epoch 4 | Batch 51 | Loss: 0.013325
Epoch 4 | Train: 0.015726 | Val: 0.015198 | Gap: -0.000528
Epoch 5 | Batch 1 | Loss: 0.014391
Epoch 5 | Batch 51 | Loss: 0.016297
Epoch 5 | Train: 0.015567 | Val: 0.015879 | Gap: 0.000312
Epoch 6 | Batch 1 | Loss: 0.011119
Epoch 6 | Batch 51 | Loss: 0.016519
Epoch 6 | Train: 0.015111 | Val: 0.015144 | Gap: 0.000033
Epoch 7 | Batch 1 | Loss: 0.010991
Epoch 7 | Batch 51 | Loss: 0.018660
Epoch 7 | Train: 0.015218 | Val: 0.015555 | Gap: 0.000337
Epoch 8 | Batch 1 | Loss: 0.014684
Epoch 8 | Batch 51 | Loss: 0.012928
Epoch 8 | Train: 0.0146

In [69]:
from src.nano_maker.modules.nano_printer.smiles_handler import smiles_fingerprint

sample_smiles = "CC/C(=C(\c1ccc(cc1)O)/c2ccc(cc2)OCCN(C)C)/c3ccccc3"
sample_map4 = torch.tensor(smiles_fingerprint(sample_smiles), dtype=torch.float32).unsqueeze(0)
model.generate(sample_map4)

[tensor([[0.4644, 0.4205, 0.4767]], grad_fn=<AddmmBackward0>),
 tensor([[0.4995, 0.3826, 0.5092]], grad_fn=<AddmmBackward0>),
 tensor([[0.4090, 0.4602, 0.5157]], grad_fn=<AddmmBackward0>),
 tensor([[0.4325, 0.4183, 0.5183]], grad_fn=<AddmmBackward0>),
 tensor([[0.5105, 0.4413, 0.4956]], grad_fn=<AddmmBackward0>),
 tensor([[0.4501, 0.4496, 0.5266]], grad_fn=<AddmmBackward0>),
 tensor([[0.4275, 0.4213, 0.5481]], grad_fn=<AddmmBackward0>),
 tensor([[0.4922, 0.4560, 0.5220]], grad_fn=<AddmmBackward0>),
 tensor([[0.4411, 0.4228, 0.5333]], grad_fn=<AddmmBackward0>),
 tensor([[0.4917, 0.4676, 0.5083]], grad_fn=<AddmmBackward0>),
 tensor([[0.4274, 0.3482, 0.4970]], grad_fn=<AddmmBackward0>),
 tensor([[0.4825, 0.4833, 0.4929]], grad_fn=<AddmmBackward0>),
 tensor([[0.4372, 0.4198, 0.4754]], grad_fn=<AddmmBackward0>),
 tensor([[0.5181, 0.4634, 0.4737]], grad_fn=<AddmmBackward0>),
 tensor([[0.4058, 0.4301, 0.5028]], grad_fn=<AddmmBackward0>),
 tensor([[0.4726, 0.3872, 0.5068]], grad_fn=<AddmmBackw